## Lib

In [48]:
import pandas as pd
import numpy as np
import re
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import nltk
nltk.download("words", quiet=True)
from nltk.corpus import words

## Load and cleaning datasets

In [49]:
def clean_word(word):
    """Membersihkan kata slang/formal."""
    word = str(word).lower().strip()
    word = re.sub(r'\s+', ' ', word)
    return word
def is_valid_slang(word):
    """Validasi panjang kata/frasa agar tidak memakan kalimat utuh."""
    if not word or pd.isna(word):
        return False
    if len(word.split()) > 3:
        return False
    return True
poison_words = {
    "is", "am", "are", "was", "were", "be", "been", "being",
    "do", "does", "did", "have", "has", "had",
    "to", "in", "on", "at", "by", "for", "of", "or", "as", "so", "if", "it",
    "i", "me", "my", "we", "us", "you", "he", "she", "they", "them",
    "u", "r", "y", "im", "ur", "a", "an", "the", "and", "but", "with"
}
slang_df = pd.read_csv("datasets/slang.csv")
slang1 = slang_df["acronym"].dropna().apply(clean_word)
slang1 = slang1[~slang1.isin(poison_words)]
genz_df = pd.read_csv("datasets/gen_zz_words.csv")
slang2 = genz_df["Word/Phrase"].dropna().apply(clean_word)
slang2 = slang2[~slang2.isin(poison_words)]
genz_slang_df = pd.read_csv("datasets/genz_slang.csv")
slang3 = genz_slang_df["Word"].dropna().apply(clean_word)
slang3 = slang3[~slang3.isin(poison_words)]
all_slangs_df = pd.read_csv("datasets/all_slangs.csv")
slang4 = all_slangs_df["Slang"].dropna().apply(clean_word)
slang4 = slang4[~slang4.isin(poison_words)]
all_slang_words = pd.concat([slang1, slang2, slang3, slang4])
all_slang_words = all_slang_words[all_slang_words.apply(is_valid_slang)]
slang_list = list(set(all_slang_words.tolist()))
print(f"Total posion yang di hapus: {len(slang_list)}")

Total posion yang di hapus: 5170


## add formal word

In [50]:
formal_words_raw = [w.lower() for w in words.words() if w.isalpha()]
modern_words = [
    "computer", "internet", "software", "hardware", "website",
    "project", "algorithm", "database", "programming", "coding",
    "smartphone", "laptop", "application", "system", "developer"
]
formal_words_raw.extend(modern_words)
formal_words_raw = list(set(formal_words_raw))
print(f"Total initial formal words: {len(formal_words_raw)}")
# filter kata formal dari slang = formal - slang
slang_set = set(slang_list)
formal_list = [w for w in formal_words_raw if w not in slang_set]
print(f"Total formal words after filtering slang collisions: {len(formal_list)}")
# balancing dataset
np.random.seed(42)
np.random.shuffle(formal_list)
formal_list_balanced = formal_list[:len(slang_list)]
print(f"Positif (Slang) : {len(slang_list)}")
print(f"Negatif (Formal): {len(formal_list_balanced)}")

Total initial formal words: 234383
Total formal words after filtering slang collisions: 233904
Positif (Slang) : 5170
Negatif (Formal): 5170


---
## 4. Create DataFrame & Train/Test Split

In [51]:
X = slang_list + formal_list_balanced
y = [1] * len(slang_list) + [0] * len(formal_list_balanced)
dataset = pd.DataFrame({"word": X, "label": y})
X_train, X_test, y_train, y_test = train_test_split(
    dataset["word"],
    dataset["label"],
    test_size=0.2,
    random_state=42,
    stratify=dataset["label"]
)
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

Train size: 8272 | Test size: 2068


---
## 5. Model Evaluation: 5-Fold Cross Validation

In [52]:
ngram_ranges = [(2, 4), (2, 5), (3, 5)]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("Cross Validation Result")
best_f1 = 0
best_ngram = None
for ngram in ngram_ranges:
    f1_scores = []
    acc_scores = []
    #tfidf
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_cv_train, X_cv_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_cv_train, y_cv_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        vec = TfidfVectorizer(analyzer="char", ngram_range=ngram)
        X_cv_train_vec = vec.fit_transform(X_cv_train)
        X_cv_val_vec = vec.transform(X_cv_val)
        model = LogisticRegression(max_iter=5000, random_state=42)
        model.fit(X_cv_train_vec, y_cv_train)
        preds = model.predict(X_cv_val_vec)
        f1_scores.append(f1_score(y_cv_val, preds))
        acc_scores.append(accuracy_score(y_cv_val, preds))
    avg_f1 = np.mean(f1_scores)
    avg_acc = np.mean(acc_scores)
    print(f"N-Gram {ngram}: Avg Accuracy = {avg_acc:.4f} | Avg F1 Score = {avg_f1:.4f}")
    if avg_f1 > best_f1:
        best_f1 = avg_f1
        best_ngram = ngram
print(f"\nBest N-Gram configuration based on F1 Score: {best_ngram}")

Cross Validation Result
N-Gram (2, 4): Avg Accuracy = 0.8992 | Avg F1 Score = 0.8962
N-Gram (2, 5): Avg Accuracy = 0.8926 | Avg F1 Score = 0.8877
N-Gram (3, 5): Avg Accuracy = 0.8933 | Avg F1 Score = 0.8947

Best N-Gram configuration based on F1 Score: (2, 4)


---
## 6. Final Model Training & Evaluation

In [53]:
# Train final vectorizer and model using the best ngram on the FULL training set
final_vectorizer = TfidfVectorizer(analyzer="char", ngram_range=best_ngram)
X_train_vec = final_vectorizer.fit_transform(X_train)
X_test_vec = final_vectorizer.transform(X_test)
final_model = LogisticRegression(max_iter=5000, random_state=42)
final_model.fit(X_train_vec, y_train)
y_pred = final_model.predict(X_test_vec)
print("Final Evaluation test")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_pred):.4f}\n")
print("Classification Report")
print(classification_report(y_test, y_pred, target_names=["Formal (0)", "Slang (1)"]))
print("Confusion Matrix")
cm = confusion_matrix(y_test, y_pred)
print(f"TN: {cm[0][0]} | FP: {cm[0][1]}")
print(f"FN: {cm[1][0]} | TP: {cm[1][1]}")

Final Evaluation test
Accuracy  : 0.9023
Precision : 0.9306
Recall    : 0.8694
F1 Score  : 0.8990

Classification Report
              precision    recall  f1-score   support

  Formal (0)       0.88      0.94      0.91      1034
   Slang (1)       0.93      0.87      0.90      1034

    accuracy                           0.90      2068
   macro avg       0.90      0.90      0.90      2068
weighted avg       0.90      0.90      0.90      2068

Confusion Matrix
TN: 967 | FP: 67
FN: 135 | TP: 899


---
## 7. Build Slang Normalization Dictionary

Kita memisahkan SLANG DETECTION (Logistic Regression) dengan SLANG NORMALIZATION.
Untuk mendapatkan hasil akhir yang natural, kita bangun dictionary hanya dari:
1. **Custom Dictionary** (Prioritas utama, untuk slang populer dengan makna natural)
2. **slang.csv** (Prioritas kedua, karena formatnya `acronym -> expansion` yang natural)

*Catatan: dataset lain seperti genz_slang.csv berisi definisi panjang (Meaning/Description), sehingga diabaikan agar output tidak memuat definisi melainkan kalimat formal yang natural.*

In [54]:
custom_dict = {
    "rizz": "charisma",
    "sigma": "confident person",
    "fire": "amazing",
    "lowkey": "kind of",
    "highkey": "definitely",
    "finna": "going to",
    "frfr": "for real",
    "no cap": "honestly",
    "sus": "suspicious",
    "dank": "excellent",
    "w": "win",
    "l": "loss",
    "bruh": "bro",
    "goat": "greatest of all time",
    "ngl" : "not gonna lie",
    "plz" : "please",
    "ur" : "you are",
    "chk": "check",
    "cnt": "cant",
    "rn": "right now",
    "af": "as fuck",
    "fyi": "for your information",
    "tbh": "to be honest",
    "bsy": "busy",
    "abt": "about",
    "bussin": "amazing"
}
final_slang_dict = {}
for _, row in slang_df.dropna(subset=["acronym", "expansion"]).iterrows():
    acro = clean_word(row["acronym"])
    if acro in poison_words:
        continue
    raw_exp = str(row["expansion"]).lower()
    first_exp = raw_exp.split(',')[0].split('/')[0].strip()
    exp = re.sub(r'\s+', ' ', first_exp)
    if acro and exp:
        final_slang_dict[acro] = exp
for k, v in custom_dict.items():
    final_slang_dict[k] = v
print(f"Total normalization entries: {len(final_slang_dict)}")

Total normalization entries: 3218


---
## 8. Save Artifacts

In [55]:
os.makedirs("models", exist_ok=True)

with open("models/slang_classifier.pkl", "wb") as f:
    pickle.dump(final_model, f)

with open("models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(final_vectorizer, f)

with open("models/slang_dictionary.pkl", "wb") as f:
    pickle.dump(final_slang_dict, f)

print("Saving models")

Saving models
